# IRIS Prediction Explorer

Interactive UI to explore model predictions, especially worst cases.

In [ ]:
import sys
import os
from pathlib import Path

# Set project root (parent of notebooks/)
PROJECT_ROOT = Path(os.getcwd()).parent
if PROJECT_ROOT.name != 'iris':
    # Fallback: maybe already at project root
    PROJECT_ROOT = Path(os.getcwd())
    if PROJECT_ROOT.name != 'iris':
        # Hardcode for JupyterHub
        PROJECT_ROOT = Path('/dscilab_hvch261_tuanminh/workspace/code/mantis/iris')

print(f"Project root: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Try importing widgets (optional for interactive mode)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")
    HAS_WIDGETS = False

from models.iris import IRIS
from datasets.dataset import IRISDataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Load Model and Data

In [ ]:
# Config paths (relative to PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / 'configs/qata.yaml'
CHECKPOINT_PATH = PROJECT_ROOT / 'output/checkpoints/qata_fusion_h8_e4/best.pth'
DICE_RESULTS_PATH = PROJECT_ROOT / 'output/analysis/dice_distribution/dice_results.json'

# Load config
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Resolve data paths relative to PROJECT_ROOT (config uses paths relative to project root)
test_dir = (PROJECT_ROOT / cfg['data']['test_dir']).resolve()
print(f"Test dir: {test_dir}")
print(f"Exists: {test_dir.exists()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load model
model = IRIS(
    visual_encoder='resnet',
    text_encoder='clinicalbert',
    fusion='bifmd',
    num_classes=1,
    img_size=cfg['data']['image_size'],
    base_channel=64,
    num_heads=8,
    expansion=4,
).to(device)

# Load checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
state = ckpt.get('model', ckpt.get('model_state_dict', ckpt))
model.load_state_dict(state, strict=False)
model.eval()
print("Model loaded!")

In [ ]:
# Load dataset with resolved absolute path
dataset = IRISDataset(
    root_dir=str(test_dir),
    image_size=cfg['data']['image_size'],
    augment=False,
)
print(f"Dataset: {len(dataset)} samples")

# Load dice results
with open(DICE_RESULTS_PATH) as f:
    dice_results = json.load(f)

# Sort by dice
dice_results_sorted = sorted(dice_results, key=lambda x: x['dice'])
print(f"Dice results loaded: {len(dice_results_sorted)} samples")

## Visualization Function

In [ ]:
def visualize_sample(idx, show_prediction=True):
    """Visualize a single sample with GT and prediction."""
    sample = dataset[idx]
    img = sample['image']
    mask = sample['mask']
    text = sample['text']
    
    # Get prediction
    if show_prediction:
        with torch.no_grad():
            img_tensor = img.unsqueeze(0).to(device)
            pred = model(img_tensor, [text])
            if isinstance(pred, dict):
                pred = pred['pred']
            pred_mask = (torch.sigmoid(pred) > 0.5).float().cpu()[0, 0]
            pred_prob = torch.sigmoid(pred).cpu()[0, 0]
    
    # Convert image for display
    img_np = img.permute(1, 2, 0).numpy()
    mask_np = mask.squeeze().numpy()
    
    # Compute dice
    if show_prediction:
        intersection = (pred_mask.numpy() * mask_np).sum()
        union = pred_mask.numpy().sum() + mask_np.sum()
        dice = 2 * intersection / (union + 1e-8)
    
    # Plot
    fig, axes = plt.subplots(1, 4 if show_prediction else 2, figsize=(16, 4))
    
    # Original image
    axes[0].imshow(img_np)
    axes[0].set_title('Input Image')
    axes[0].axis('off')
    
    # GT mask
    axes[1].imshow(img_np)
    axes[1].imshow(mask_np, cmap='Reds', alpha=0.5)
    axes[1].set_title(f'Ground Truth (size: {mask_np.sum():.0f})')
    axes[1].axis('off')
    
    if show_prediction:
        # Prediction mask
        axes[2].imshow(img_np)
        axes[2].imshow(pred_mask.numpy(), cmap='Blues', alpha=0.5)
        axes[2].set_title(f'Prediction (Dice: {dice*100:.1f}%)')
        axes[2].axis('off')
        
        # Probability map
        axes[3].imshow(pred_prob.numpy(), cmap='hot', vmin=0, vmax=1)
        axes[3].set_title('Probability Map')
        axes[3].axis('off')
    
    plt.suptitle(f'Text: "{text}"', fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return dice if show_prediction else None

## Interactive Explorer

In [ ]:
if not HAS_WIDGETS:
    print("Install ipywidgets for interactive mode: pip install ipywidgets")
else:
    # Create dropdown options from sorted dice results
    options = [(f"[{r['dice']*100:5.1f}%] {r['image_id'][:50]}", r['idx']) 
               for r in dice_results_sorted]

    # Widgets
    sample_dropdown = widgets.Dropdown(
        options=options,
        description='Sample:',
        layout=widgets.Layout(width='600px')
    )

    range_slider = widgets.IntRangeSlider(
        value=[0, 50],
        min=0,
        max=100,
        step=5,
        description='Dice Range:',
        layout=widgets.Layout(width='400px')
    )

    filter_btn = widgets.Button(description='Filter by Range')
    output = widgets.Output()

    def on_sample_change(change):
        with output:
            clear_output(wait=True)
            idx = change['new']
            visualize_sample(idx)

    def on_filter_click(b):
        low, high = range_slider.value
        filtered = [(f"[{r['dice']*100:5.1f}%] {r['image_id'][:50]}", r['idx']) 
                    for r in dice_results_sorted 
                    if low <= r['dice']*100 < high]
        sample_dropdown.options = filtered if filtered else [('No samples in range', -1)]

    sample_dropdown.observe(on_sample_change, names='value')
    filter_btn.on_click(on_filter_click)

    # Display
    display(widgets.HBox([range_slider, filter_btn]))
    display(sample_dropdown)
    display(output)

    # Show first sample
    with output:
        visualize_sample(dice_results_sorted[0]['idx'])

## Quick View: Worst N Samples

In [ ]:
# View worst 10
print("=" * 60)
print("WORST 10 SAMPLES")
print("=" * 60)

for i, r in enumerate(dice_results_sorted[:10]):
    print(f"\n--- Sample {i+1}: Dice = {r['dice']*100:.1f}% ---")
    visualize_sample(r['idx'])

## Analyze Patterns in Worst Cases

In [ ]:
# Analyze text patterns in worst 100 samples
worst_100 = dice_results_sorted[:100]

from collections import Counter

# Count keywords
keywords = ['unilateral', 'bilateral', 'one', 'two', 'three', 
            'left', 'right', 'upper', 'middle', 'lower']

print("Keyword frequency in WORST 100 samples:")
for kw in keywords:
    count = sum(1 for r in worst_100 if kw in r['text'].lower())
    print(f"  {kw:12s}: {count:3d} ({count}%)")

print("\nKeyword frequency in ALL samples:")
for kw in keywords:
    count = sum(1 for r in dice_results if kw in r['text'].lower())
    pct = count / len(dice_results) * 100
    print(f"  {kw:12s}: {count:4d} ({pct:.1f}%)")